# XWebAgent User Study — Analysis

**Steps**
1. Run `python download_data.py` once to populate `data/`
2. Run cells top-to-bottom

Or skip section 2 and call `download_data.main()` directly from this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.contingency_tables import mcnemar
from urllib.parse import urlparse
import json, os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

COND_COLORS = {'control': '#E57373', 'extension': '#64B5F6'}
TASK_ORDER  = ['find', 'guide', 'hide']
os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)
print('Ready')

## 1. Download from Supabase

In [ ]:
import requests
try:
    from config import SUPABASE_URL, SUPABASE_ANON_KEY
except ImportError:
    SUPABASE_URL      = 'https://YOUR_PROJECT.supabase.co'
    SUPABASE_ANON_KEY = 'YOUR_ANON_KEY'

def fetch_table(table):
    headers = {'apikey': SUPABASE_ANON_KEY,
               'Authorization': f'Bearer {SUPABASE_ANON_KEY}'}
    rows, offset = [], 0
    while True:
        resp = requests.get(
            f'{SUPABASE_URL}/rest/v1/{table}?select=*&limit=1000&offset={offset}',
            headers=headers, timeout=30)
        if resp.status_code == 401:
            print(f'401 on {table}: check RLS / key')
            return pd.DataFrame()
        resp.raise_for_status()
        batch = resp.json()
        rows.extend(batch)
        if len(batch) < 1000: break
        offset += 1000
    return pd.DataFrame(rows)

sessions = fetch_table('study_sessions')
tasks    = fetch_table('study_task_results')
sessions.to_csv('data/sessions.csv', index=False)
tasks.to_csv('data/tasks.csv', index=False)
print(f'Sessions: {len(sessions)}  |  Task results: {len(tasks)}')

## 2. Load & preprocess

In [ ]:
# Load (re-run this cell any time to reload from CSV)
sessions = pd.read_csv('data/sessions.csv')
tasks    = pd.read_csv('data/tasks.csv')

# Parse JSONB columns
def safe_json(x):
    if isinstance(x, (list, dict)): return x
    try: return json.loads(x) if pd.notna(x) else []
    except: return []

for col in ['chat_transcript', 'page_visit_urls', 'task_data', 'user_hidden_selectors']:
    if col in tasks.columns:
        tasks[col] = tasks[col].apply(safe_json)

tasks['answer_correct'] = tasks['answer_correct'].map(
    {True: True, False: False, 'true': True, 'false': False,
     't': True, 'f': False, 1: True, 0: False})

tasks['time_s']   = tasks['time_ms'].astype(float) / 1000
tasks['time_min'] = tasks['time_s'] / 60

for col in ['scroll_count','ctrl_f_count','text_select_count',
            'page_visit_count','chat_turn_count','hidden_count',
            'click_count','mouse_move_px']:
    if col in tasks.columns:
        tasks[col] = pd.to_numeric(tasks[col], errors='coerce').fillna(0).astype(int)

# Compute hide_recall from user_hidden_selectors vs task_data.hidden_elements
# when the column is absent or all-null (older data)
if 'hide_recall' not in tasks.columns:
    tasks['hide_recall'] = np.nan
tasks['hide_recall'] = pd.to_numeric(tasks['hide_recall'], errors='coerce')

def compute_hide_recall(row):
    td = row.get('task_data') if isinstance(row.get('task_data'), dict) else {}
    gt = set(td.get('hidden_elements', []))
    user_sel = row.get('user_hidden_selectors', [])
    if not isinstance(user_sel, list): user_sel = []
    user = set(user_sel)
    if not gt: return np.nan
    return len(gt & user) / len(gt)

hide_missing = tasks['task_type'].eq('hide') & tasks['hide_recall'].isna()
if hide_missing.any() and 'user_hidden_selectors' in tasks.columns:
    tasks.loc[hide_missing, 'hide_recall'] = tasks.loc[hide_missing].apply(compute_hide_recall, axis=1)

print(f'Participants: {tasks["participant_id"].nunique()}')
print(f'Task results: {len(tasks)}')
tasks.groupby(['task_type','condition']).size().unstack(fill_value=0)

## 3. Participant overview

In [ ]:
summary = (tasks.groupby('participant_id')
           .agg(n_tasks=('id', 'count'),
                conditions=('condition', lambda x: ' | '.join(sorted(x.unique()))),
                task_types=('task_type', lambda x: ', '.join(sorted(x.unique()))))
           .reset_index())
display(summary)

# Condition order per participant
if not sessions.empty:
    display(sessions[['participant_id','condition_order','created_at']])

## 4. Task completion time

In [ ]:
# Summary table
display(tasks.groupby(['task_type','condition'])['time_s']
        .agg(mean='mean', median='median', std='std', n='count').round(1))

# Box + strip plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ttype in zip(axes, TASK_ORDER):
    sub = tasks[tasks['task_type'] == ttype]
    sns.boxplot(data=sub, x='condition', y='time_s', palette=COND_COLORS,
                order=['control','extension'], ax=ax, width=0.5, fliersize=0)
    sns.stripplot(data=sub, x='condition', y='time_s',
                  order=['control','extension'],
                  color='#333', alpha=0.5, size=5, ax=ax, jitter=True)
    ax.set_title(ttype.capitalize(), fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Time (s)' if ttype == 'find' else '')
fig.suptitle('Task Completion Time — Control vs Extension', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/time_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Paired slope chart
paired = (tasks.pivot_table(index=['participant_id','task_type'],
                             columns='condition', values='time_s', aggfunc='mean')
          .reset_index()
          .dropna(subset=['control','extension']))

if not paired.empty:
    paired['diff_s']      = paired['control'] - paired['extension']
    paired['speedup_pct'] = (paired['diff_s'] / paired['control'] * 100).round(1)
    display(paired.groupby('task_type')[['diff_s','speedup_pct']].mean().round(2))

    fig, ax = plt.subplots(figsize=(8, 5))
    task_colors = {'find': '#2196F3', 'guide': '#4CAF50', 'hide': '#FF9800'}
    for _, row in paired.iterrows():
        c = task_colors.get(row['task_type'], '#888')
        ax.plot(['Control','Extension'], [row['control'], row['extension']],
                'o-', color=c, alpha=0.6, linewidth=1.5, markersize=6)
    handles = [mpatches.Patch(color=c, label=t.capitalize())
               for t, c in task_colors.items()]
    ax.legend(handles=handles, title='Task type')
    ax.set_ylabel('Time (s)')
    ax.set_title('Paired Completion Times per Participant', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plots/paired_times.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Task success rates

In [ ]:
# Find accuracy
find_t = tasks[tasks['task_type'] == 'find']
if not find_t.empty and 'answer_correct' in find_t.columns:
    acc = find_t.groupby('condition')['answer_correct'].mean().mul(100).round(1)
    print('Find accuracy (%):')
    display(acc.rename('accuracy_%').to_frame())

    # Per-participant accuracy (paired)
    p_acc = (find_t.groupby(['participant_id', 'condition'])['answer_correct']
             .mean().mul(100).round(1).unstack(fill_value=np.nan))
    print('Per-participant Find accuracy (%):')
    display(p_acc)

    # Confidence × accuracy breakdown
    if 'confidence' in find_t.columns:
        ca = (find_t.groupby(['condition', 'confidence'])['answer_correct']
              .agg(n='count', correct='sum')
              .assign(pct=lambda d: (d['correct'] / d['n'] * 100).round(1))
              .reset_index())
        print('Accuracy by confidence level:')
        display(ca)

# Guide + Hide completion
other = tasks[tasks['task_type'].isin(['guide','hide'])]
if not other.empty:
    answer_order  = ['completed','partial','failed']
    answer_colors = {'completed': '#4CAF50', 'partial': '#FFC107', 'failed': '#F44336'}

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, ttype in zip(axes, ['guide','hide']):
        sub = other[other['task_type'] == ttype]
        ct  = sub.groupby(['condition','answer']).size().unstack(fill_value=0)
        ct_pct = ct.div(ct.sum(axis=1), axis=0).mul(100)
        valid = [a for a in answer_order if a in ct_pct.columns]
        ct_pct[valid].plot(kind='bar', ax=ax, stacked=True,
                           color=[answer_colors[a] for a in valid],
                           edgecolor='white', rot=0)
        ax.set_title(f'{ttype.capitalize()} Task Outcome', fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('% participants')
        ax.set_ylim(0, 100)
        ax.legend(title='Outcome', bbox_to_anchor=(1,1))
    plt.tight_layout()
    plt.savefig('plots/completion_rates.png', dpi=150, bbox_inches='tight')
    plt.show()

# Hide recall
hide_t = tasks[tasks['task_type'] == 'hide']
if not hide_t.empty and 'hide_recall' in hide_t.columns:
    has_recall = hide_t['hide_recall'].notna()
    if has_recall.any():
        print('Hide recall by condition:')
        display(hide_t[has_recall].groupby('condition')['hide_recall']
                .agg(mean='mean', median='median', std='std', n='count').mul(100).round(1))

## 6. Behavioral metrics

In [ ]:
beh_cols = {'scroll_count': 'Scroll gestures', 'ctrl_f_count': 'Ctrl+F',
            'text_select_count': 'Text selections', 'click_count': 'Mouse clicks',
            'mouse_move_px': 'Mouse distance (px)', 'page_visit_count': 'Page visits'}
available = {k: v for k, v in beh_cols.items() if k in tasks.columns}

if available:
    # Mean per condition
    display(tasks.groupby('condition')[list(available)].mean().round(2).rename(columns=available))

    # Overview bars
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (col, label) in zip(axes, available.items()):
        sns.barplot(data=tasks, x='condition', y=col,
                    order=['control','extension'], palette=COND_COLORS,
                    ax=ax, errorbar='se', capsize=0.12)
        ax.set_title(label, fontsize=11)
        ax.set_xlabel('')
        ax.set_ylabel('Mean count')
    fig.suptitle('Behavioral Metrics by Condition (mean ± SE)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plots/behavioral_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Breakdown by task type
if available:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (col, label) in zip(axes, available.items()):
        sns.barplot(data=tasks, x='task_type', y=col, hue='condition',
                    order=TASK_ORDER, hue_order=['control','extension'],
                    palette=COND_COLORS, ax=ax, errorbar='se', capsize=0.08)
        ax.set_title(label, fontsize=11)
        ax.set_xlabel('')
        ax.legend(title='', fontsize=9)
    fig.suptitle('Behavioral Metrics by Task Type', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plots/behavioral_by_task.png', dpi=150, bbox_inches='tight')
    plt.show()

# Top domains (control)
if 'page_visit_urls' in tasks.columns:
    ctrl_urls = [u for urls in tasks[tasks['condition']=='control']['page_visit_urls']
                 if isinstance(urls, list) for u in urls]
    if ctrl_urls:
        domains = pd.Series([urlparse(u).netloc.replace('www.','') for u in ctrl_urls
                              if u]).value_counts().head(15)
        fig, ax = plt.subplots(figsize=(10, 4))
        domains.plot(kind='barh', ax=ax, color='#78909C')
        ax.invert_yaxis()
        ax.set_title('Top Visited Domains — Control Condition',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('plots/top_domains.png', dpi=150, bbox_inches='tight')
        plt.show()

## 7. Chat usage (extension condition)

In [ ]:
ext = tasks[tasks['condition'] == 'extension'].copy()

if not ext.empty and 'chat_turn_count' in ext.columns:
    print('Mean queries per task type:')
    display(ext.groupby('task_type')['chat_turn_count'].describe().round(2))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    max_q = max(int(ext['chat_turn_count'].max()), 1)
    sns.histplot(data=ext, x='chat_turn_count', bins=range(0, max_q + 2),
                 hue='task_type', multiple='stack',
                 hue_order=TASK_ORDER, ax=axes[0], palette='Set2')
    axes[0].set_title('AI Queries per Task', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Number of queries')

    if 'helpfulness' in ext.columns:
        help_order  = ['very','somewhat','not','unused']
        help_colors = {'very': '#4CAF50', 'somewhat': '#8BC34A',
                       'not': '#F44336', 'unused': '#9E9E9E'}
        ct = ext.groupby(['task_type','helpfulness']).size().unstack(fill_value=0)
        ct_pct = ct.div(ct.sum(axis=1), axis=0).mul(100)
        valid = [c for c in help_order if c in ct_pct.columns]
        if valid:
            ct_pct[valid].plot(kind='bar', ax=axes[1], stacked=True,
                               color=[help_colors[c] for c in valid],
                               edgecolor='white', rot=0)
            axes[1].set_title('Helpfulness Ratings', fontsize=12, fontweight='bold')
            axes[1].set_ylabel('% participants')
            axes[1].set_ylim(0, 100)
            axes[1].legend(title='Rating', bbox_to_anchor=(1,1))

    plt.tight_layout()
    plt.savefig('plots/chat_usage.png', dpi=150, bbox_inches='tight')
    plt.show()

    corr = ext[['chat_turn_count','time_s']].corr().iloc[0,1]
    print(f'Correlation (queries vs time): r = {corr:.3f}')

## 8. Statistical tests

Within-subject design: **Wilcoxon signed-rank** (non-parametric, suitable for small n) 
and **paired t-test** for comparison.
Effect size = Cohen's d on the pairwise differences.

In [ ]:
n_participants = tasks['participant_id'].nunique()
if n_participants < 5:
    print(f'⚠️  Only {n_participants} participants — results are illustrative, not conclusive.\n')

records = []

# ── Completion time ──────────────────────────────────────────────────────────
print('[ Completion Time ]')
for ttype in TASK_ORDER:
    sub    = tasks[tasks['task_type'] == ttype]
    paired = (sub.pivot_table(index='participant_id', columns='condition',
                              values='time_s', aggfunc='mean')
              .dropna(subset=['control','extension']))
    if len(paired) < 3 or 'control' not in paired.columns or 'extension' not in paired.columns:
        print(f'  {ttype}: n={len(paired)} — skip')
        continue

    ctrl, ext_vals = paired['control'].values, paired['extension'].values
    diff = ctrl - ext_vals
    try:
        _, p_w = wilcoxon(ctrl, ext_vals)
    except Exception:
        p_w = float('nan')
    _, p_t  = ttest_rel(ctrl, ext_vals)
    cohen_d = diff.mean() / diff.std() if diff.std() > 0 else float('nan')
    sig     = '**' if p_w < 0.05 else ('†' if p_w < 0.1 else 'ns')

    print(f'  {ttype:6s}  n={len(paired):2d}  ctrl={ctrl.mean():.1f}s  '
          f'ext={ext_vals.mean():.1f}s  diff={diff.mean():+.1f}s  '
          f'W p={p_w:.4f} {sig}  t p={p_t:.4f}  d={cohen_d:.2f}')
    records.append({'metric': 'time_s', 'task_type': ttype, 'n': len(paired),
                    'mean_ctrl': round(float(ctrl.mean()),2),
                    'mean_ext': round(float(ext_vals.mean()),2),
                    'mean_diff': round(float(diff.mean()),2),
                    'wilcoxon_p': round(float(p_w),4),
                    'ttest_p': round(float(p_t),4),
                    'cohens_d': round(float(cohen_d),3)})

print('  ** p<0.05  † p<0.10  ns not significant\n')

# ── Find accuracy: McNemar's test ────────────────────────────────────────────
find_t = tasks[tasks['task_type'] == 'find'].copy()
if not find_t.empty and 'answer_correct' in find_t.columns:
    print("[ Find Accuracy — McNemar's test ]")
    mc_data = find_t[['participant_id','condition','answer_correct']].dropna()
    piv = mc_data.pivot_table(index='participant_id', columns='condition',
                              values='answer_correct', aggfunc='first').dropna()
    if 'control' in piv.columns and 'extension' in piv.columns:
        n_11 = ((piv['control']==True)  & (piv['extension']==True)).sum()
        n_10 = ((piv['control']==True)  & (piv['extension']==False)).sum()
        n_01 = ((piv['control']==False) & (piv['extension']==True)).sum()
        n_00 = ((piv['control']==False) & (piv['extension']==False)).sum()
        table = [[n_11, n_10], [n_01, n_00]]
        try:
            result = mcnemar(table, exact=True)
            print(f'  McNemar exact p = {result.pvalue:.4f}')
        except Exception as e:
            print(f'  McNemar test skipped: {e}')
        print(f'  both_correct={n_11}  ctrl_only={n_10}  ext_only={n_01}  both_wrong={n_00}\n')

# ── Hide recall ──────────────────────────────────────────────────────────────
hide_t = tasks[tasks['task_type'] == 'hide']
if not hide_t.empty and 'hide_recall' in hide_t.columns and hide_t['hide_recall'].notna().any():
    print('[ Hide Recall ]')
    paired_r = (hide_t.dropna(subset=['hide_recall'])
                .pivot_table(index='participant_id', columns='condition',
                             values='hide_recall', aggfunc='mean')
                .dropna(subset=['control','extension']))
    if len(paired_r) >= 2 and 'control' in paired_r.columns and 'extension' in paired_r.columns:
        c_r, e_r = paired_r['control'].values, paired_r['extension'].values
        try:
            _, p_w_r = wilcoxon(c_r, e_r)
        except Exception:
            p_w_r = float('nan')
        print(f'  ctrl={c_r.mean():.3f}  ext={e_r.mean():.3f}  '
              f'diff={c_r.mean()-e_r.mean():+.3f}  Wilcoxon p={p_w_r:.4f}\n')

if records:
    stats_df = pd.DataFrame(records)
    stats_df.to_csv('data/stats_results.csv', index=False)
    display(stats_df)

## 9. Export clean summary CSV

In [ ]:
keep = ['participant_id','session_id','block_index','task_index','question_index',
        'task_type','condition','time_s','answer','answer_correct','confidence','helpfulness',
        'chat_turn_count','hidden_count','hide_recall',
        'scroll_count','ctrl_f_count','text_select_count',
        'click_count','mouse_move_px',
        'page_visit_count','question_or_task']
cols = [c for c in keep if c in tasks.columns]
out  = tasks[cols].copy()
out.to_csv('data/summary.csv', index=False)
print(f'Saved data/summary.csv  ({len(out)} rows)')
display(out.head(6))

## 8. Guide screenshots

In [ ]:
import base64
from io import BytesIO
from PIL import Image

guide_rows = tasks[
    tasks['task_type'].eq('guide') & tasks['guide_screenshot'].notna() &
    tasks['guide_screenshot'].ne('')
].reset_index(drop=True) if 'guide_screenshot' in tasks.columns else pd.DataFrame()

if guide_rows.empty:
    print('No guide screenshots available.')
else:
    print(f'{len(guide_rows)} guide screenshot(s) found.')
    images, titles = [], []
    for _, row in guide_rows.iterrows():
        val = row['guide_screenshot']
        try:
            if isinstance(val, str) and os.path.isfile(val):
                img = Image.open(val)
            else:
                raw = val.split(',', 1)[-1] if ',' in val else val
                img = Image.open(BytesIO(base64.b64decode(raw)))
            images.append(img)
            pid  = row.get('participant_id', '?')
            cond = row.get('condition', '?')
            qi   = row.get('question_index', row.get('task_index', '?'))
            task = str(row.get('question_or_task', ''))[:40]
            titles.append(f'{pid} | {cond} | q{qi}\n{task}')
        except Exception as e:
            print(f'  Could not load screenshot: {e}')

    if images:
        cols = min(4, len(images))
        rows_n = (len(images) + cols - 1) // cols
        fig, axes = plt.subplots(rows_n, cols, figsize=(5*cols, 4*rows_n))
        axes_flat = np.array(axes).reshape(-1) if rows_n * cols > 1 else [axes]
        for ax, img, title in zip(axes_flat, images, titles):
            ax.imshow(img)
            ax.set_title(title, fontsize=8)
            ax.axis('off')
        for ax in axes_flat[len(images):]:
            ax.axis('off')
        fig.suptitle('Guide Task Screenshots', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('plots/guide_screenshots.png', dpi=150, bbox_inches='tight')
        plt.show()